<a href="https://colab.research.google.com/github/Jaychandrarevada/Innomatics-Internship-Tasks/blob/main/NLP_Task_5_Fine_Tuning_BERT_for_POS_Tagging_%26_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
!pip install -q transformers datasets evaluate seqeval

In [50]:
from datasets import load_dataset

# Task 1: Dataset Selection
# Use the clean Parquet version of CoNLL-2003
dataset = load_dataset("lhoestq/conll2003")

# Manually define the 23 standard chunk tags for CoNLL-2003 to bypass the Parquet schema error
label_list = [
    "O", "B-ADJP", "I-ADJP", "B-ADVP", "I-ADVP", "B-CONJP", "I-CONJP",
    "B-INTJ", "I-INTJ", "B-LST", "I-LST", "B-NP", "I-NP", "B-PP", "I-PP",
    "B-PRT", "I-PRT", "B-SBAR", "I-SBAR", "B-UCP", "I-UCP", "B-VP", "I-VP"
]

# Task 3 requirement: Proper label mapping
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print(f"Number of labels: {len(label_list)}")
print(f"Labels: {label_list}")

Number of labels: 23
Labels: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [51]:
from transformers import AutoTokenizer

# Task 3: Use DistilBERT (faster and lighter for Colab) [cite: 56]
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Task 2: Align labels with tokens [cite: 47]
def tokenize_and_align_labels(examples):
    # Tokenize using BERT/DistilBERT tokenizer [cite: 46]
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens get -100 [cite: 50]
            if word_idx is None:
                label_ids.append(-100)
            # Only label the first subword of a given word [cite: 49]
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Subsequent subwords get -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    # Expected Outputs: input_ids, attention_mask, labels [cite: 51, 52, 53, 54]
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply preprocessing to the entire dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print("Tokenization and label alignment complete!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Tokenization and label alignment complete!


In [52]:
import evaluate
import numpy as np

# Use seqeval metric [cite: 69]
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and convert to string labels
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    # Report required metrics [cite: 70, 71, 72, 73]
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }
print("Metrics setup complete!")

Metrics setup complete!


In [55]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

# Task 3: Use AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list), # Correct num_labels
    id2label=id2label,          # Proper label mapping
    label2id=label2id           # Proper label mapping
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Task 4: Define Training Arguments
args = TrainingArguments(
    output_dir="./distilbert-chunking",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Task 4: Use Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer, # UPDATED: Changed from 'tokenizer' to 'processing_class'
)

# Train the model
print("Starting training...")
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.459613,0.199844,0.906425,0.901776,0.904094,0.949671
2,0.165662,0.178263,0.912223,0.909352,0.910786,0.953468


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.459613,0.199844,0.906425,0.901776,0.904094,0.949671
2,0.165662,0.178263,0.912223,0.909352,0.910786,0.953468
3,0.132823,0.172668,0.914579,0.911878,0.913226,0.954791


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=2634, training_loss=0.21634268054658026, metrics={'train_runtime': 286.2415, 'train_samples_per_second': 147.159, 'train_steps_per_second': 9.202, 'total_flos': 510251380802730.0, 'train_loss': 0.21634268054658026, 'epoch': 3.0})

In [56]:
from transformers import pipeline

# Task 6: Load trained model [cite: 74, 75]
# The aggregation_strategy="simple" helps group subwords back together nicely!
chunker = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Predict on custom sentences [cite: 76]
custom_text = "John works at Google in California" # [cite: 77]
predictions = chunker(custom_text)

print(f"Input: {custom_text}\n") # [cite: 77]
print("Chunk Tags Output: \n") # [cite: 80]
for pred in predictions:
    print(f"Word: {pred['word']} | Chunk: {pred['entity_group']} | Score: {pred['score']:.4f}")

Input: John works at Google in California

Chunk Tags Output: 

Word: john | Chunk: NP | Score: 0.9970
Word: works | Chunk: VP | Score: 0.7482
Word: at | Chunk: PP | Score: 0.9858
Word: google | Chunk: NP | Score: 0.9809
Word: in | Chunk: PP | Score: 0.9958
Word: california | Chunk: NP | Score: 0.9938


## Final Report: Fine-Tuning DistilBERT for Chunking

### 1. POS Tagging vs. Chunking (Task 7)
[cite_start]While both are token classification tasks, they operate at different levels of complexity[cite: 82, 87]:
* **POS Tagging (Grammar-level):** This task assigns a single grammatical label (Noun, Verb, Adjective) to each word. [cite_start]It is generally considered "Easy" because a model usually only needs local context (the words immediately before and after) to determine the correct tag[cite: 83].
* **Chunking (Phrase-level):** Also known as shallow parsing, chunking groups consecutive words into meaningful syntactic phrases (e.g., Noun Phrases, Prepositional Phrases). [cite_start]This is a "Medium" difficulty task because the model must understand both the grammatical role of the words and the boundaries of the phrase (using BIO tags to mark the Beginning, Inside, and Outside of phrases)[cite: 84].

### 2. Challenges Faced (Task 8)
* **Library Deprecations:** During dataset loading, we encountered strict security updates in the Hugging Face `datasets` library that disabled dataset scripts (`trust_remote_code`). We had to pivot to a clean, pre-converted Parquet version of CoNLL-2003 (`lhoestq/conll2003`) to bypass this. [cite_start]Furthermore, updates to the `Trainer` API required swapping deprecated arguments like `tokenizer` for `processing_class`[cite: 88].
* **Label Alignment for Subwords:** Because DistilBERT uses WordPiece tokenization, single words are often split into multiple subwords. [cite_start]We had to implement a custom function to align our dataset labels with these new tokens, assigning the `-100` index to special tokens and secondary subwords so they wouldn't skew the model's loss calculation during training[cite: 48, 49, 50, 88].

### 3. Observations & Insights
* Transformer models like DistilBERT are incredibly powerful for sequence labeling. [cite_start]Even with a small learning rate and just 3 epochs, the model quickly learned to identify complex phrase boundaries[cite: 89].
* [cite_start]Utilizing the `seqeval` metric is crucial for sequence tasks, as standard accuracy is misleading when the majority of tokens might be simple "O" (Outside) tags[cite: 69]. [cite_start]Precision, Recall, and F1 provide a much clearer picture of how well the model is identifying actual entities[cite: 70, 71, 72, 73, 89].